In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(base_options=base_options, num_faces=1)
landmarker = vision.FaceLandmarker.create_from_options(options)

cam = cv2.VideoCapture(0)

while True:
    ret, frame = cam.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_image)

    if result.face_landmarks:
        for landmarks in result.face_landmarks:
            for lm in landmarks:
                x = int(lm.x * frame.shape[1])
                y = int(lm.y * frame.shape[0])
                cv2.circle(frame, (x, y), 1, (0, 255, 0), -1)

    cv2.imshow("Face Mesh", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()

In [2]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(base_options=base_options, num_faces=1)
landmarker = vision.FaceLandmarker.create_from_options(options)

cam = cv2.VideoCapture(0)

# iris landmark indices
LEFT_IRIS = [468, 469, 470, 471, 472]
RIGHT_IRIS = [473, 474, 475, 476, 477]

while True:
    ret, frame = cam.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_image)

    if result.face_landmarks:
        landmarks = result.face_landmarks[0]

        for idx in LEFT_IRIS:
            lm = landmarks[idx]
            x, y = int(lm.x * frame.shape[1]), int(lm.y * frame.shape[0])
            cv2.circle(frame, (x, y), 3, (255, 0, 0), -1)

        for idx in RIGHT_IRIS:
            lm = landmarks[idx]
            x, y = int(lm.x * frame.shape[1]), int(lm.y * frame.shape[0])
            cv2.circle(frame, (x, y), 3, (0, 0, 255), -1)

    cv2.imshow("Iris Tracking", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()

W0000 00:00:1784520878.657538 11685441 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1784520878.663316 11685441 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1784520878.666088 11685447 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784520878.676892 11685447 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
E0000 00:00:1784520878.679321 11683219 portable_clearcut_uploader.cc:90] Failed to send to clearcut: FAILED_PRECONDITION: Not valid for uploading until: 2026-07-20T00:28:24.231293-04:00
=== Source Location Trace: === 
wireless/android/play/playlog/cplusplus/portable_clearcut_uploader.cc:180


KeyboardInterrupt: 

In [4]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(base_options=base_options, num_faces=1)
landmarker = vision.FaceLandmarker.create_from_options(options)

LEFT_IRIS = [468, 469, 470, 471, 472]

# 5 calibration points: corners + center of screen
screen_w, screen_h = 1280, 720
calibration_points = [
    (100, 100), (screen_w - 100, 100),
    (screen_w // 2, screen_h // 2),
    (100, screen_h - 100), (screen_w - 100, screen_h - 100),
]

iris_positions = []
cam = cv2.VideoCapture(0)

for point in calibration_points:
    print(f"Look at the dot and press SPACE when ready...")
    while True:
        ret, frame = cam.read()
        if not ret:
            continue

        display = np.zeros((screen_h, screen_w, 3), dtype=np.uint8)
        cv2.circle(display, point, 20, (0, 255, 0), -1)
        cv2.imshow("Calibration", display)

        key = cv2.waitKey(1) & 0xFF
        if key == ord(' '):
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            result = landmarker.detect(mp_image)
            if result.face_landmarks:
                landmarks = result.face_landmarks[0]
                iris_x = np.mean([landmarks[i].x for i in LEFT_IRIS])
                iris_y = np.mean([landmarks[i].y for i in LEFT_IRIS])
                iris_positions.append((iris_x, iris_y))
                print(f"Captured point {point}: iris at ({iris_x:.3f}, {iris_y:.3f})")
                break

cam.release()
cv2.destroyAllWindows()

iris_positions = np.array(iris_positions)
calibration_points = np.array(calibration_points)

print("Calibration complete")
print(iris_positions)
print(calibration_points)

W0000 00:00:1784521012.728887 11687539 face_landmarker_graph.cc:180] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
I0000 00:00:1784521012.734499 11687539 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4
W0000 00:00:1784521012.736733 11687543 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1784521012.748723 11687544 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Look at the dot and press SPACE when ready...
Captured point (100, 100): iris at (0.491, 0.390)
Look at the dot and press SPACE when ready...
Captured point (1180, 100): iris at (0.459, 0.390)
Look at the dot and press SPACE when ready...
Captured point (640, 360): iris at (0.474, 0.398)
Look at the dot and press SPACE when ready...
Captured point (100, 620): iris at (0.491, 0.407)
Look at the dot and press SPACE when ready...
Captured point (1180, 620): iris at (0.461, 0.406)
Calibration complete
[[0.49125401 0.38953257]
 [0.45937774 0.39008698]
 [0.47352077 0.39791043]
 [0.49118841 0.40724121]
 [0.46129135 0.40643981]]
[[ 100  100]
 [1180  100]
 [ 640  360]
 [ 100  620]
 [1180  620]]


In [7]:
pip install scikit-learn

  Using cached scikit_learn-1.9.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached scipy-1.18.0-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.24.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp313-cp313-macosx_12_0_arm64.whl (8.2 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached narwhals-2.24.0-py3-none-any.whl (461 kB)
Using cached scipy-1.18.0-cp313-cp313-macosx_14_0_arm64.whl (20.4 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [scikit-learn]0m 4/5 [scikit-learn]

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
from sklearn.linear_model import LinearRegression

model_x = LinearRegression()
model_y = LinearRegression()

model_x.fit(iris_positions, calibration_points[:, 0])
model_y.fit(iris_positions, calibration_points[:, 1])

print(f"X model R²: {model_x.score(iris_positions, calibration_points[:, 0]):.3f}")
print(f"Y model R²: {model_y.score(iris_positions, calibration_points[:, 1]):.3f}")

NameError: name 'iris_positions' is not defined

In [5]:
cam = cv2.VideoCapture(0)

while True:
    ret, frame = cam.read()
    if not ret:
        break

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = landmarker.detect(mp_image)

    display = np.zeros((screen_h, screen_w, 3), dtype=np.uint8)

    if result.face_landmarks:
        landmarks = result.face_landmarks[0]
        iris_x = np.mean([landmarks[i].x for i in LEFT_IRIS])
        iris_y = np.mean([landmarks[i].y for i in LEFT_IRIS])

        pred_x = int(model_x.predict([[iris_x, iris_y]])[0])
        pred_y = int(model_y.predict([[iris_x, iris_y]])[0])

        pred_x = np.clip(pred_x, 0, screen_w)
        pred_y = np.clip(pred_y, 0, screen_h)

        cv2.circle(display, (pred_x, pred_y), 15, (0, 255, 0), -1)

    cv2.imshow("Gaze Prediction", display)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()

NameError: name 'model_x' is not defined